In [4]:
import tqdm
import torch
import numpy as np
from bgflow.utils import as_numpy
from bgflow import DiffEqFlow, BoltzmannGenerator, MeanFreeNormalDistribution
from bgflow import BlackBoxDynamics, BruteForceEstimator
from tbg.models2 import EGNN_dynamics_AD2_cat


n_particles = 22
n_dimensions = 3
dim = n_particles * n_dimensions

scaling = 10

n_particles = 22
n_dimensions = 3
dim = n_particles * n_dimensions


# atom types for backbone
atom_types = np.array(
    [1, 0, 0, 0, 4, 3, 5, 0, 6, 0, 1, 0, 0, 0, 7, 3, 8, 0, 1, 0, 0, 0]
)
h_initial = torch.nn.functional.one_hot(torch.tensor(atom_types))


# now set up a prior
prior = MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False).cuda()
prior_cpu = MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False)

brute_force_estimator = BruteForceEstimator()
net_dynamics = EGNN_dynamics_AD2_cat(
    n_particles=n_particles,
    device="cuda",
    n_dimension=dim // n_particles,
    h_initial=h_initial,
    hidden_nf=64,
    act_fn=torch.nn.SiLU(),
    n_layers=5,
    recurrent=True,
    tanh=True,
    attention=True,
    condition_time=True,
    mode="egnn_dynamics",
    agg="sum",
)

bb_dynamics = BlackBoxDynamics(
    dynamics_function=net_dynamics, divergence_estimator=brute_force_estimator
)

flow = DiffEqFlow(dynamics=bb_dynamics)

bg = BoltzmannGenerator(prior, flow, prior).cuda()


class BruteForceEstimatorFast(torch.nn.Module):
    """
    Exact bruteforce estimation of the divergence of a dynamics function.
    """

    def __init__(self):
        super().__init__()
        self.n = 0

    def forward(self, dynamics, t, xs):
        self.n += 1

        with torch.set_grad_enabled(True):
            xs.requires_grad_(True)
            x = [xs[:, [i]] for i in range(xs.size(1))]

            dxs = dynamics(t, torch.cat(x, dim=1))

            assert len(dxs.shape) == 2, "`dxs` must have shape [n_btach, system_dim]"
            divergence = 0
            for i in range(xs.size(1)):
                divergence += torch.autograd.grad(
                    dxs[:, [i]], x[i], torch.ones_like(dxs[:, [i]]), retain_graph=True
                )[0]

        return dxs, -divergence.view(-1, 1)


brute_force_estimator_fast = BruteForceEstimatorFast()
# use OTD in the evaluation process
bb_dynamics._divergence_estimator = brute_force_estimator_fast
bg.flow._integrator_atol = 1e-4
bg.flow._integrator_rtol = 1e-4
flow._use_checkpoints = False
flow._kwargs = {}

filename = "Flow-Matching-AD2-amber-weighted-backbone"

PATH_last = f"models/{filename}"
checkpoint = torch.load(PATH_last)
flow.load_state_dict(checkpoint["model_state_dict"])


/tmp/ipykernel_3770400/1742305686.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH_last)


<All keys matched successfully>

In [5]:
# alpha_t = t
# dot alpha_t = 1
# beta_t = 1 - t
# dot beta_t = -1
# kappa_t = 1 / t
# eta_t = (1 - t) ( 1 / t * (1-t) - (-1))
# eta_t = (1 - t) ((1 - t) / t + t / t)
# eta_t = (1 - t) (1 / t)
# eta_t = (1 - t) / t

# f = 1 / t
# score = 1 / eta (vt - xt / t)

In [6]:
def stochastic_flow(t, xt):
    vt = flow_fn(t, xt)
    return 2 * vt - xt / torch.clip(t, min=1e-2)

In [5]:
flow_fn = flow._dynamics._dynamics._dynamics_function

In [15]:
%%time
import pdb

filename = "Flow-Matching-AD2-amber-weighted-encoding-SDE-2"
n_samples = 400
n_sample_batches = 10
latent_np = np.empty(shape=(0))
samples_np = np.empty(shape=(0))
dlogp_np = np.empty(shape=(0))
print(f"Start sampling with {filename}")
for i in tqdm.tqdm(range(n_sample_batches)):
    
    x0 = prior.sample(400)
    xt = x0
    #logp = prior.energy(x0)
    logp = torch.zeros(x0.shape[0], device="cuda")
    eps = 1e-3
    with torch.no_grad():
        n_steps = 1000
        dt = 1 / n_steps
        for t in torch.linspace(0, 1, n_steps + 1):
            
            sigma_t_squared = (2 * (1 - t) / torch.clip(t, min=eps))
            # sigma_t = (2 (1 - t) / t) ** 0.5
            sigma_t = sigma_t_squared ** 0.5
            vt = flow._dynamics._dynamics._dynamics_function(t, xt)
            # st is correct we checked
            st = 2 * vt - xt / torch.clip(t, min=1e-2)
            noise_t = sigma_t * torch.randn_like(xt) * (dt ** 0.5)
            dxt = st * dt + noise_t
            xt = xt + dxt
            score_t = (t * vt - xt) / torch.clip(1 - t, min=eps)
            dlogp = ((-xt.shape[-1] / torch.clip(t, min=eps) 
                      + (score_t * (-sigma_t_squared / 2 * score_t + dxt - xt / torch.clip(t, min=eps))).sum(-1)) * dt)
            logp = logp + dlogp
        
        #print("n / i = ", flow_fn.counter / (i+1))
        latent_np = np.append(latent_np, x0.detach().cpu().numpy())
        samples_np = np.append(samples_np, xt.detach().cpu().numpy())
        
        dlogp_np = np.append(dlogp_np, as_numpy(logp))

    latent_np = latent_np.reshape(-1, dim)
    samples_np = samples_np.reshape(-1, dim)
    np.savez(
        f"result_data/{filename}",
        latent_np=latent_np,
        samples_np=samples_np,
        dlogp_np=dlogp_np,
    )


Start sampling with Flow-Matching-AD2-amber-weighted-encoding-SDE-2


100%|█████████████████████████████████████████████████████████████| 10/10 [02:37<00:00, 15.74s/it]

CPU times: user 2min 35s, sys: 80.8 ms, total: 2min 35s
Wall time: 2min 37s


In [3]:
x0_base = prior.sample(400)

In [3]:
x0_base = prior.sample(400)

In [10]:
%%time
import pdb

filename = "Flow-Matching-AD2-amber-weighted-encoding-ODE-4"
n_samples = 400
n_sample_batches = 50
latent_np = np.empty(shape=(0))
samples_np = np.empty(shape=(0))
dlogp_np = np.empty(shape=(0))
print(f"Start sampling with {filename}")
for i in tqdm.tqdm(range(n_sample_batches)):
    
    x0 = prior.sample(400)
    xt = x0
    #logp = prior.energy(x0)
    logp = torch.zeros(x0.shape[0], device="cuda")
    eps = 1e-3
    with torch.no_grad():
        n_steps = 1
        dt = 1 / n_steps

        
        #print(y.shape)
        #break
        for t in torch.linspace(0, 1, n_steps + 1):
            
            sigma_t_squared = (2 * (1 - t) / torch.clip(t, min=eps))
            # sigma_t = (2 (1 - t) / t) ** 0.5
            sigma_t = sigma_t_squared ** 0.5
            vt = flow_fn(t, xt)
            # st is correct we checked
            st = 2 * vt - xt / torch.clip(t, min=1e-2)
            noise_t = sigma_t * torch.randn_like(xt) * (dt ** 0.5)
            dxt = vt * dt + noise_t * 0
            xt = xt + dxt
            score_t = (t * vt - xt) / torch.clip(1 - t, min=eps)
            dlogp = ((-xt.shape[-1] / torch.clip(t, min=eps) 
                      + (score_t * (-sigma_t_squared / 2 * score_t + dxt - xt / torch.clip(t, min=eps))).sum(-1)) * dt)
            logp = 0 # logp + dlogp
        from torchdiffeq import odeint_adjoint
        ts = torch.linspace(0, 1, 2).cuda()
        xt = odeint_adjoint(flow._dynamics._dynamics._dynamics_function, x0, ts, method="dopri5", rtol=1e-4, atol=1e-4)[-1]
        #print("n / i = ", flow_fn.counter / (i+1))
        latent_np = np.append(latent_np, x0.detach().cpu().numpy())
        samples_np = np.append(samples_np, xt.detach().cpu().numpy())
        dlogp_np = np.append(dlogp_np, as_numpy(logp))

    latent_np = latent_np.reshape(-1, dim)
    samples_np = samples_np.reshape(-1, dim)
    np.savez(
        f"result_data/{filename}",
        latent_np=latent_np,
        samples_np=samples_np,
        dlogp_np=dlogp_np,
    )


Start sampling with Flow-Matching-AD2-amber-weighted-encoding-ODE-4


100%|█████████████████████████████████████████████████████████████| 50/50 [01:12<00:00,  1.45s/it]

CPU times: user 1min 11s, sys: 619 ms, total: 1min 12s
Wall time: 1min 12s


In [7]:
filename = "Flow-Matching-AD2-amber-weighted-encoding-Base-backbone"
n_sample_batches = 50
n_samples=400
latent_np = np.empty(shape=(0))
samples_np = np.empty(shape=(0))
dlogp_np = np.empty(shape=(0))
for i in tqdm.tqdm(range(n_sample_batches)):
    with torch.no_grad():
        samples, latent, dlogp = bg.sample(n_samples, with_latent=True, with_dlogp=True)
        print("n / i = ", flow_fn.counter / (i+1))
        latent_np = np.append(latent_np, latent.detach().cpu().numpy())
        samples_np = np.append(samples_np, samples.detach().cpu().numpy())

        dlogp_np = np.append(dlogp_np, as_numpy(dlogp))

    latent_np = latent_np.reshape(-1, dim)
    samples_np = samples_np.reshape(-1, dim)
    print(samples_np.std())
    np.savez(
        f"result_data/{filename}",
        latent_np=latent_np,
        samples_np=samples_np,
        dlogp_np=dlogp_np,
    )
    break

/network/scratch/a/alexander.tong/micromamba/envs/tbg2/lib/python3.10/site-packages/torchdiffeq/_impl/misc.py:15: UserWarning: Dopri5Solver: Unexpected arguments {'temperature': 1.0}
  warnings.warn('{}: Unexpected arguments {}'.format(solver.__class__.__name__, unused_kwargs))
/home/mila/a/alexander.tong/active_inference/tbg/tbg/models2.py:395: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t = torch.tensor(t).to(xs)
  0%|                                                                      | 0/50 [01:34<?, ?it/s]

n / i =  128.0
1.6303805946161911


In [11]:
samples_np.std()

np.float64(1.5710516092086217)

In [ ]:
}